# Exploring the book catalogue

Notebook 01 produced `data/raw/books.csv`, one row per book. This notebook turns that table
into answers to three plain questions: how prices are spread, whether higher-rated books cost
more, and which titles look like the best value. Along the way we build a compact summary
table and the same figures the [dashboard](../app/README.md) serves.

## Learning objectives

By the end of this notebook you can:

- Load a saved table with the shared `ds_practice` loader.
- Inspect shape, types, and missing values before analysing anything.
- Summarise a distribution with the mean, median, and interquartile range.
- Compare groups with a grouped table and a bar chart.
- Read a correlation with care when the variable is an ordinal rating.
- Define a custom score ("value") and explain its weaknesses.

## Concept

A tidy table is one row per observation (here, a book) and one column per variable. That shape
is what makes pandas useful: `describe()` summarises a column, `groupby()` compares groups,
and `corr()` measures a relationship between two numeric columns.

Exploratory analysis is a short list of concrete questions, not a hunt for a story. We ask
them in order, look at the numbers, and let the answer be "no relationship" if that is what
the data says. Two cautions apply throughout:

- **Ratings are ordinal.** "4 stars" is not twice "2 stars", so a correlation treats them as
  numbers only as a rough convenience.
- **A "value" score is a choice.** `rating / price` makes cheap books look good almost by
  construction. We state the definition and check that the ranking is sensible.

We load through [`ds_practice`](../../src/ds_practice) rather than re-reading the CSV by hand, so
every module uses the same loader and the same plotting theme. The loader raises a helpful
error if the file is missing; we catch it and print the fetch command instead of crashing.

## Worked example

We load the scraped catalogue, check its shape and types, then work through the three
questions with a mix of summary tables and figures. Every calculation is guarded so the
notebook still runs, with a clear message, when the CSV is absent.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ds_practice import (
    barplot,
    boxplot,
    histogram,
    load_books,
    scatterplot,
    set_seed,
    set_theme,
)

set_seed(42)
set_theme()

try:
    books = load_books()
except FileNotFoundError as exc:
    books = None
    print("books.csv was not found.")
    print(exc)
    print("Fetch it with: python scripts/download_data.py --module 05")

if books is not None:
    print("shape:", books.shape)
    display(books.head())

### Step 1 — Inspect before trusting

Types and missing values first. If `rating` arrived as text or `price_gbp` had gaps, every
later calculation would be suspect.

In [ ]:
if books is not None:
    print(books.dtypes)
    print()
    print("missing values:")
    print(books.isna().sum())
    print()
    print("stock status counts:")
    print(books["in_stock"].value_counts())
else:
    print("skipping: no data loaded")

### Step 2 — How are prices distributed?

A summary gives the numbers; a histogram shows the shape. The mean is pulled by a few
expensive books, so we read it next to the median.

In [ ]:
if books is not None:
    price = books["price_gbp"]
    print(price.describe().round(2))
    print("median:", round(price.median(), 2))
    q1, q3 = price.quantile([0.25, 0.75])
    print("IQR:", round(q3 - q1, 2))

    fig, ax = histogram(
        price, bins=15, title="Distribution of book prices", xlabel="Price (GBP)"
    )
    plt.show()
else:
    print("skipping: no data loaded")

### Step 3 — Do higher-rated books cost more?

We compare the mean and median price for each star rating, then measure the correlation
between rating and price. Ratings are ordinal, so the mean is a convenience; the median and
the counts guard against a group dominated by one or two books.

In [ ]:
if books is not None:
    by_rating = (
        books.groupby("rating")["price_gbp"]
        .agg(count="count", mean="mean", median="median")
        .round(2)
    )
    display(by_rating)

    fig, ax = barplot(
        by_rating.index,
        by_rating["mean"],
        title="Mean price by star rating",
        xlabel="Rating (stars)",
        ylabel="Mean price (GBP)",
    )
    plt.show()

    correlation = books["rating"].corr(books["price_gbp"])
    print(f"Pearson correlation between rating and price: {correlation:.3f}")
else:
    print("skipping: no data loaded")

The correlation is close to zero: on this sandbox, rating and price are essentially unrelated.
That is a real finding, not a failed analysis — a plausible-sounding relationship can simply be
absent from the data.

### Step 4 — A compact summary table

One table that can answer most quick questions about the catalogue: how many books sit at each
rating, what they cost, and how wide the spread is.

In [ ]:
if books is not None:
    summary = pd.DataFrame(
        {
            "books": books.groupby("rating").size(),
            "mean_price": books.groupby("rating")["price_gbp"].mean(),
            "median_price": books.groupby("rating")["price_gbp"].median(),
            "min_price": books.groupby("rating")["price_gbp"].min(),
            "max_price": books.groupby("rating")["price_gbp"].max(),
        }
    ).round(2)
    summary.index.name = "rating"
    display(summary)
else:
    print("skipping: no data loaded")

### Step 5 — What would "best value" mean?

There is no single definition of value. Here we use rating per pound and list the top five.
Because the score favours cheap books, the scatter of price against rating is a useful cross
check on whether the winners are genuinely well rated.

In [ ]:
if books is not None:
    books["value"] = books["rating"] / books["price_gbp"]
    best = books.nlargest(5, "value")[["title", "rating", "price_gbp", "value"]].round(2)
    display(best)

    fig, ax = scatterplot(
        books["price_gbp"],
        books["rating"],
        title="Rating against price",
        xlabel="Price (GBP)",
        ylabel="Rating (stars)",
    )
    plt.show()
else:
    print("skipping: no data loaded")

### Step 6 — The dashboard reads the same file

The [`app/dashboard.py`](../app/dashboard.py) dashboard loads the identical CSV and recomputes
`value` itself, so there is a single source of truth. Nothing derived needs to be exported.

In [ ]:
if books is not None:
    print("columns in the raw table:", list(books.columns))
    print("rows ready for the dashboard:", len(books))
else:
    print("skipping: no data loaded")

## Exercises

1. **Describe the spread.** Report the count, mean, median, and IQR of `price_gbp`, and count
   how many books sit above the 90th percentile. Add a boxplot to show the same spread
   visually.
2. **Price by rating.** For each rating, report the count, mean, and median price. Which
   rating is the most expensive on average, and how large is the gap to the cheapest?
3. **Rethink value.** Recompute `value` as `(rating / 5) / price_gbp`. Do the top five titles
   change compared with `rating / price_gbp`? Explain why or why not. (Worked answers are in
   [`../solutions.md`](../solutions.md).)

## Limitations

The catalogue is small (about 100 books after a five-page scrape) and entirely fictional, so
its prices and ratings say nothing about real books. "Value" is an invented score, not a
recommendation, and it treats an ordinal rating as a number. The correlation is a simple
Pearson coefficient computed on that same ordinal variable; it is a description of this table,
not evidence of cause and effect. Finally, the table is a single snapshot with no history, so
it cannot show how prices or ratings change.